# 🔵 Unsupervised Learning: K-Means, DBSCAN, PCA
---
- **K-Means:** Group similar items into K clusters
- **DBSCAN:** Density-based, finds any shape, detects outliers
- **PCA:** Reduce dimensions keeping max information

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
warnings.filterwarnings('ignore')

df = pd.read_csv('loan_default_dataset.csv')
for c in df.select_dtypes(include=[np.number]).columns:
    df[c].fillna(df[c].median(), inplace=True)
for c in df.select_dtypes(include=['object']).columns:
    df[c].fillna(df[c].mode()[0], inplace=True)
    df = pd.concat([df.drop(c, axis=1), pd.get_dummies(df[c], prefix=c, drop_first=True)], axis=1)

X = df.drop('Default', axis=1) if 'Default' in df.columns else df
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'✅ Data ready: {X_scaled.shape}')

## 🔵 K-Means Clustering
**Analogy:** Sort mixed balls into K groups by color similarity.
1. Pick K random centers
2. Assign each point to nearest center
3. Move center to middle of its group
4. Repeat until stable

In [ ]:
# ELBOW METHOD: Find best K
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2)
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method - Pick K at the elbow!', fontweight='bold')
plt.tight_layout()
plt.show()
print('👆 Look for the bend (elbow) in the curve!')

In [ ]:
# Apply K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_km = kmeans.fit_predict(X_scaled)
print(f'K-Means Clusters: {dict(Counter(clusters_km))}')

# Visualize with PCA (reduce to 2D)
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters_km, cmap='viridis', alpha=0.5, s=15)
plt.title('K-Means Clusters (K=3)', fontweight='bold')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()

## 🔵 DBSCAN
**Analogy:** Find crowded areas = clusters. Lonely points = outliers.
- eps: How close is nearby
- min_samples: Min neighbors to form group
- No need to specify K!
- Detects outliers automatically!

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=10)
clusters_db = dbscan.fit_predict(X_2d)

n_clusters = len(set(clusters_db)) - (1 if -1 in clusters_db else 0)
n_noise = list(clusters_db).count(-1)
print(f'DBSCAN: {n_clusters} clusters, {n_noise} outliers')

plt.figure(figsize=(10, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters_db, cmap='viridis', alpha=0.5, s=15)
plt.title(f'DBSCAN: {n_clusters} clusters, {n_noise} noise points', fontweight='bold')
plt.colorbar(label='Cluster (-1 = Noise/Outlier)')
plt.tight_layout()
plt.show()

## 📉 PCA (Principal Component Analysis)
**Analogy:** 3D photo → best 2D shadow keeping max info.
100 features → 10 features (95% info kept!)

In [ ]:
pca_full = PCA(n_components=0.95, random_state=42)
X_pca = pca_full.fit_transform(X_scaled)

print(f'Original features: {X_scaled.shape[1]}')
print(f'PCA features (95% variance): {X_pca.shape[1]}')
print(f'Reduction: {X_scaled.shape[1] - X_pca.shape[1]} features removed!')

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(10, 5))
plt.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
        pca_full.explained_variance_ratio_, alpha=0.6, label='Individual')
plt.step(range(1, len(cumvar)+1), cumvar, where='mid', color='red', linewidth=2, label='Cumulative')
plt.axhline(0.95, color='gray', linestyle='--', label='95%')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.title('PCA - Variance Explained', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## K-Means vs DBSCAN Summary
| Feature | K-Means | DBSCAN |
|---------|---------|--------|
| Need K? | Yes | No |
| Cluster shape | Spherical | Any shape |
| Outliers | No | Yes (noise) |
| Speed | Fast | Slower |
| Parameters | K | eps, min_samples |